In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import time
import pyarrow
import fastparquet
from datetime import datetime
# Part 1: Extraction

# ==========================================
# PHASE 0: CONFIGURATION & GLOBALS
# ==========================================
API_KEY = "56f0398519088075aac607773115b9e0"
BASE_URL = "https://auto-poster.co.uk/yt_api"

HEADERS = {
    'AP_API_KEY': API_KEY
}

# Time Boundaries: March 1, 2026 to May 8, 2026
START_TIMESTAMP = int(datetime(2026, 3, 1).timestamp())
END_TIMESTAMP = int(datetime(2026, 5, 8).timestamp())
TARGET_ACCOUNT = "jewishoncampus"

In [ ]:
def test_phase_1_posts(target_user):
    """
    Tests the feed endpoint to verify the JSON structure for posts.
    """
    print(f"\n--- TESTING PHASE I: POSTS FOR '{target_user}' ---")
    API_URL = f"{BASE_URL}/get_ig_user_posts.php"
    payload = {

        "username_or_url": target_user
    }

    response = requests.post(API_URL, data=payload, headers=HEADERS)

    if response.status_code == 200:
        print(json.dumps(response.json(), indent=4))
    else:
        print(f"Error {response.status_code}: {response.text}")

test_phase_1_posts(TARGET_ACCOUNT)

In [ ]:
def test_phase_1_mapping(target_user):
    """
    Validates the 'posts -> node -> id/code/taken_at' JSON structure.
    """
    print(f"\n--- VERIFYING PHASE I JSON MAPPING FOR '{target_user}' ---")

    # The confirmed POST endpoint
    API_URL = f"{BASE_URL}/get_ig_user_posts.php"

    # Form-data payload
    payload = {
        "username_or_url": target_user
    }

    # 1. Execute the POST request
    response = requests.post(API_URL, data=payload, headers=HEADERS)

    if response.status_code == 200:
        data = response.json()

        # 2. Target the main 'posts' array
        posts_array = data.get('posts', [])

        if not posts_array:
            print("Error: The 'posts' array is empty or nested differently. Printing full JSON:")
            print(json.dumps(data, indent=4))
            return

        print(f"✅ Successfully located the 'posts' array! Found {len(posts_array)} posts on page 1.\n")

        # 3. Isolate the first post to test the 'node' extraction
        first_post = posts_array[14]
        node = first_post.get('node', {})

        if not node:
            print("Error: Could not find the 'node' key. Printing the first post's JSON:")
            print(json.dumps(first_post, indent=4))
            return

        # 4. Extract the critical variables
        post_id = node.get('id')
        media_code = node.get('code')
        post_ts = node.get('taken_at')

        # Format the timestamp for easy visual confirmation
        human_date = "Unknown"
        if post_ts:
            human_date = datetime.fromtimestamp(post_ts).strftime('%Y-%m-%d %H:%M:%S')

        # 5. Print the parsed results
        print("--- EXTRACTION RESULTS FOR POST #1 ---")
        print(f"Post ID:    {post_id}")
        print(f"Media Code: {media_code}")
        print(f"Timestamp:  {post_ts} ({human_date})\n")

        # Print just the node's JSON to keep your terminal clean, but allow for a visual check
        print("--- RAW JSON OF THE NODE (For final verification) ---")
        print(json.dumps(node, indent=4))

    else:
        print(f"Error {response.status_code}: {response.text}")

# Execution: test_phase_1_mapping(TARGET_ACCOUNT)

In [ ]:
def test_phase_2_comments(test_post_id, test_media_code):
    """
    Validates the GET request and reveals the JSON schema for comments.
    """
    print(f"\n--- VERIFYING PHASE II COMMENTS FOR CODE '{test_media_code}' ---")

    # The confirmed GET endpoint
    API_URL = f"{BASE_URL}/get_post_comments.php"
    # Querystring for GET requests

    querystring = {
        "post_id": test_post_id,
        "media_code": test_media_code,
        "pagination_token": "", # Blank for page 1
        "sort_order": "popular"
    }

    # Execute the GET request
    response = requests.get(API_URL, headers=HEADERS, params=querystring)

    if response.status_code == 200:
        data = response.json()
        print("✅ Request Successful! Here is the comment schema:")

        # Pretty-print the exact structure so you can see where the text, ID, and pagination live
        print(json.dumps(data, indent=4))
    else:
        print(f"Error {response.status_code}: {response.text}")

# Execution: test_phase_2_comments("id", "code")

In [ ]:
# PHASE I: PROFILE EXTRACTION (POST) FUNCTION

def get_user_posts_direct(username, start_ts, end_ts):
    """Paginates through feed to extract post IDs and codes via POST."""
    valid_posts = []
    pagination_token = ""
    has_next = True

    API_URL = f"{BASE_URL}/get_ig_user_posts.php"

    while has_next:
        payload = {
            "username_or_url": username,
            "pagination_token": pagination_token
        }

        response = requests.post(API_URL, data=payload, headers=HEADERS).json()
        posts_array = response.get('posts', [])

        for index, post in enumerate(posts_array):
            node = post.get('node', {})

            # EXACT MAPPING: Using the confirmed 'taken_at' key
            post_ts = node.get('taken_at')

            # Safety net just in case a post is corrupted
            if not post_ts:
                print(f"  [DEBUG] Skipping Post {index}: 'taken_at' key missing/empty.")
                continue

            # Convert milliseconds to seconds if necessary
            if isinstance(post_ts, int) and post_ts > 3000000000:
                post_ts = int(post_ts / 1000)

            # Date Boundary Logic
            if post_ts > end_ts:
                # Post is too new
                continue

            if post_ts < start_ts:
                # Pinned Post bypass
                if not pagination_token:
                    print(f"  [DEBUG] Ignored old post on Page 1 (Likely a Pinned Post).")
                    continue
                else:
                    # Genuinely reached the chronological end
                    print(f"\nReached chronological boundary: Posts older than {datetime.fromtimestamp(start_ts)}. Phase I complete.")
                    return valid_posts

            # Valid post within the timeframe!
            valid_posts.append({
                "post_id": node.get('id'),
                "media_code": node.get('code')
            })

        page_info = response.get('page_info', {})
        has_next = page_info.get('has_next_page', False)
        pagination_token = page_info.get('end_cursor', "")

        time.sleep(1)
    return valid_posts

# PHASE II: COMMENT EXTRACTION (GET) FUNCTION

def get_comments_direct(post_id, media_code):
    """Paginates through comments using top-level stringified tokens via GET."""
    all_comments = []
    pagination_token = None
    has_next = True

    API_URL = f"{BASE_URL}/get_post_comments.php"

    while has_next:
        querystring = {
            "post_id": post_id,
            "media_code": media_code,
            "sort_order": "chronological"
        }

        if pagination_token:
            querystring["pagination_token"] = pagination_token

        response = requests.get(API_URL, headers=HEADERS, params=querystring).json()
        comments_array = response.get('comments', [])

        for comment in comments_array:
            all_comments.append({
                "post_id": post_id,
                "media_code": media_code,
                "text": comment.get('text'),
                "comment_id": comment.get('id', 'unknown'),
                "timestamp": comment.get('created_at', 'unknown'),
                "commenter_username": comment.get('user', {}).get('username', 'unknown')
            })

        new_token = response.get('pagination_token')

        if new_token:
            pagination_token = new_token
            has_next = True
        else:
            has_next = False

        time.sleep(1.5)
    return all_comments

# EXTRACTION EXECUTION SCRIPT

def run_extraction_pipeline():
    """Executes Phase I and II, then strictly saves the RAW data to disk."""
    print(f"\n--- STARTING EXTRACTION PIPELINE FOR '{TARGET_ACCOUNT}' ---")

    final_dataset = []

    print("Phase I: Fetching post details...")
    posts_to_scrape = get_user_posts_direct(TARGET_ACCOUNT, START_TIMESTAMP, END_TIMESTAMP)

    if not posts_to_scrape:
        print("No posts found in the specified timeframe. Exiting extraction.")
        return False

    print(f"Phase II: Extracting comments for {len(posts_to_scrape)} posts...")
    for post in posts_to_scrape:
        comments = get_comments_direct(post['post_id'], post['media_code'])
        final_dataset.extend(comments)

    if final_dataset:
        df = pd.DataFrame(final_dataset)
        raw_filename = f"{TARGET_ACCOUNT}_raw_comments.parquet"

        # Save securely to disk without any transformation
        df.to_parquet(raw_filename, index=False)
        print(f"Download successful! {len(df)} raw comments saved to {raw_filename}.")
        return raw_filename
    else:
        print("Extraction finished, but no comments were found.")
        return False

run_extraction_pipeline()

In [ ]:
# TRUE EXECUTION

RUN_DUMMY_TESTS_ONLY = False

if RUN_DUMMY_TESTS_ONLY:
    # ... (Dummy tests) ...
    pass

else:
    # --- EXTRACT ---
    final_dataset = []
    print(f"Phase I: Fetching post details for {TARGET_ACCOUNT}...")
    posts_to_scrape = get_user_posts_direct(TARGET_ACCOUNT, START_TIMESTAMP, END_TIMESTAMP)

    print("Phase II: Extracting comments...")
    for post in posts_to_scrape:
        comments = get_comments_direct(post['post_id'], post['media_code'])
        final_dataset.extend(comments)

    if final_dataset:
        df = pd.DataFrame(final_dataset)

        raw_filename = f"{TARGET_ACCOUNT}_RAW_comments.parquet"
        df.to_parquet(raw_filename, index=False)
        print(f"Saved {len(df)} RAW comments to {raw_filename}.")
    else:
        print("No comments were extracted.")